# Can a molecule change but not fall apart? Find the change in an amino acid's spectra

You'll tell an amino acid that stays whole from one that changes without breaking down.

Every amino acid has the same **backbone** (the part that links into a protein chain) and its own
**side chain**. Most amino acids keep their backbone in concentrated sulfuric acid. Some change
their side chain. Here you learn to tell the two apart.

Boxes headed **RECORD YOUR ANSWER** are where you write something. Being wrong or unsure is
fine: you compare with the paper at the end.

| Part | What you do | Time |
|---|---|---|
| 1 | Read a stable amino acid (worked) | 10 min |
| 2 | Predict, then load your case | 10 min |
| 3 | Find the backbone signal | 15 min |
| 4 | Write your verdict and compare | 20 min |

Data: M. D. Seager, S. Seager, Bains and Petkowski, Zenodo [10.5281/zenodo.8381013](https://doi.org/10.5281/zenodo.8381013), CC BY 4.0. Paper: *Astrobiology* 24, 386 (2024).

In [ ]:
# Setup. In Colab this first installs the training package (about 30 seconds).
import importlib.util, os
if importlib.util.find_spec("nmrtrain") is None:
    %pip install -q "nmrtrain @ git+https://github.com/Rakiura-Inc/morning-star-nmr@main"
import matplotlib.pyplot as plt
import nmrtrain as nt

NOTEBOOK = "notebooks/03_amino_acid_side_chains.ipynb"
MODE = nt.portal.connect(NOTEBOOK)

## Part 1: A stable one, worked

The next cell stacks alanine in 98% acid at 1 day, 1 week and 1 month. Its alpha hydrogen (on the
backbone carbon next to the nitrogen, about 4.6 ppm) and its methyl (about 2.0 ppm) do not move.

In [ ]:
ala = nt.load_series("8381013", "ALA", acid="K", nucleus="1H")
fig_ala, ax = nt.plot.stack(ala, 0.5, 9.5, norm=(0.5, 9.5), title="alanine in 98% acid")
plt.show()
for s in ala:
    pos, _ = nt.peaks.linewidth_hz(s, 6.82, 0.1)
    h = max((p["height"] for p in nt.peaks.pick(s, 0.5, 9.5) if abs(p["ppm"] - 6.82) < 0.05), default=0)
    print(f"{s.time_label:8s} NH3+ signal near {pos:.2f} ppm, height {h:.2f}")

### Trap: fading NH signals

The signal near 6.8 ppm shrinks. Those are NH3+ hydrogens, and they swap with deuterium from the
acid. A fading NH signal is not breakdown.

## Part 2: Predict, then load your case

Asparagine and glutamine each carry an amide (C(=O)NH2) at the end of the side chain. Both were
recorded in 81% acid.

### What you need

Most amino acids keep their backbone in this acid: 19 of 20 after 4 weeks (bootcamp step 2). A few
change only their side chain. Asparagine and glutamine have the same backbone as alanine, which you
just saw survive. What differs is the side chain, which ends in an amide group, C(=O)NH2. A peptide
bond is also an amide: you saw one break in Ala-Ala in notebook 01.

Longer version: NMR project step 2, Can a molecule change but not fall apart? Find the change in an amino acid's spectra.

> **RECORD YOUR ANSWER: predict what 81% acid does to asparagine and glutamine, and pick your case**
>
> **What gets saved:** `MY_PREDICTION`, one call for each, saved with the time, and `CASE`, the one you will read.
>
> **Where to look:** Part 1 above, and what you saw happen to Ala-Ala's peptide bond in notebook 01.
>
> **How to fill this in:**
> - `"ASN-H"` (asparagine) and `"GLN-H"` (glutamine): `"survives"`, `"side chain changed"`, `"breaks down"` or `"unsure"`
> - `CASE`: `"ASN-H"` or `"GLN-H"`, the one you will read
> - The cell after it stacks your case and prints its peak list at each time point.

<details><summary>Stuck? Open hint 1</summary>

Which part of these molecules is new compared with alanine?

</details>

<details><summary>Stuck? Open hint 2</summary>

The backbone is the same as alanine's, which survives. So the question is whether the amide
at the end of the side chain changes. Guess "survives" if you think it holds, "side chain changed"
if you think it reacts, or "unsure".

</details>

In [ ]:
MY_PREDICTION = {
    "ASN-H": ...,   # asparagine in 81% acid: "survives", "side chain changed", "breaks down" or "unsure"
    "GLN-H": ...,   # glutamine in 81% acid: the same choices
}
nt.record.predict("amino-acid", MY_PREDICTION)

CASE = "ASN-H"   # the one you will read: "ASN-H" or "GLN-H"
nt.record.saved("CASE", CASE, "the amino acid you will read");

In [ ]:
sample, acid = CASE.split("-")
ser = nt.load_series("8381013", sample, acid=acid, nucleus="1H")
fig_case, ax = nt.plot.stack(ser, 3.0, 6.5, norm=(3.0, 6.5),
                             title=f"{nt.io.NAMES[sample]} in {nt.io.ACID_PCT[acid]} acid")
plt.show()
traps = nt.checks.run_traps(ser, norm=(3.0, 6.5), lo=3.0, hi=6.5)
my_peaks = {s.time: nt.peaks.pick(s, 0.5, 9.5) for s in ser}
for t, pk in my_peaks.items():
    print(f"{t:7s}", [p["ppm"] for p in pk])
nt.portal.done("amino-acid-stack", figures=[fig_case]);

## Part 3: Find the backbone

### What you need

The backbone signal is the alpha hydrogen: the hydrogen on the carbon next to the nitrogen. In this
acid it is usually the highest-ppm signal from hydrogens on carbon of a simple amino acid (alanine's
sits near 4.6 ppm, its methyl near 2.0). Peaks above 6.5 ppm in these amino acids are hydrogens on
nitrogen (alanine's NH3+ sits near 6.8 ppm), not on carbon.

Longer version: bootcamp step 6, Does a single amino acid survive a month in Venus's acid? Check its spectra yourself.

> **RECORD YOUR ANSWER: which peak is the backbone signal?**
>
> **What gets saved:** `BACKBONE_PPM`, one number in ppm.
>
> **Where to look:** the 1 day peak list printed under your stack above.
>
> **How to fill this in:**
> - Write one value from the 1 day list, like `4.62`, with no quotes.

<details><summary>Stuck? Open hint 1</summary>

Cross out the peaks that are hydrogens on nitrogen first.

</details>

<details><summary>Stuck? Open hint 2</summary>

Of the peaks left, the alpha hydrogen sits highest, because its carbon carries the nitrogen.
The side-chain CH2 sits lower.

</details>

In [ ]:
BACKBONE_PPM = ...   # one value from the 1 day list above, like 4.62
nt.record.saved("BACKBONE_PPM", BACKBONE_PPM, "your backbone signal, in ppm");

In [ ]:
if nt.record.revealed("amino-acid"):
    print(nt.checks.check_backbone(BACKBONE_PPM))

Now follow every signal over the three time points. Look for two things: whether the backbone
signal stays, and whether anything grows next to it. The next cell lists the peaks at both times,
at the first only, and at the last only.

In [ ]:
first, last = ser[0], ser[-1]
changes = nt.peaks.compare(my_peaks[first.time], my_peaks[last.time])
for k, words in (("kept", "at both times"), ("lost", "at the first time only"), ("new", "at the last time only")):
    print(f"{words:24s}", ", ".join(f"{p:.2f}" for p in changes[k]) or "none")

### A filled example: alanine

Alanine from Part 1, filled in with the reasons. It shows the format; your case's values will
differ.

In [ ]:
ALA_EXAMPLE = {
    "call": "survives",          # its hydrogens on carbon stay where they were for a month
    "lost_ppm": [],              # nothing from hydrogens on carbon is gone
    "new_ppm": [],               # nothing new appears
    "kept_ppm": [4.63, 1.99],    # the alpha hydrogen and the methyl
    "product_guess": "none",
    "confidence": "high",
    "note": "The NH3+ signal near 6.8 ppm fades: hydrogens swapping with the acid's deuterium, not breakdown.",
}

### What you need

- **Lost:** a peak at 1 day that is gone, or much smaller, at 1 month.
- **New:** a peak at 1 month that was not there at 1 day. A side chain change shows as new signals
  that grow over time while the backbone signal stays.
- **Kept:** a peak at both times.
- A fading NH signal (above 6.5 ppm) is not breakdown.

Longer version: NMR project step 2, Can a molecule change but not fall apart? Find the change in an amino acid's spectra.

> **RECORD YOUR ANSWER: what changed in your amino acid?**
>
> **What gets saved:** `LOST_PPM`, `NEW_PPM` and `KEPT_PPM`, three lists of peak positions in ppm.
>
> **Where to look:** your stack above, and the three lists printed under it.
>
> **How to fill this in:**
> - `LOST_PPM`: peaks there at 1 day and gone or much smaller at 1 month, as numbers like `[1.23, 4.56]`
> - `NEW_PPM`: peaks at 1 month that were not there at 1 day
> - `KEPT_PPM`: peaks there at both times
> - Leave a list as `[]` if there were none. Alanine's record above shows the format.

<details><summary>Stuck? Open hint 1</summary>

Follow the backbone signal you chose first: is it still there at 1 month?

</details>

<details><summary>Stuck? Open hint 2</summary>

Then look next to each of your 1 day signals: has a smaller one grown nearby by 1 week, and grown
more by 1 month?

</details>

In [ ]:
LOST_PPM = []    # peaks there at 1 day and gone or much smaller at 1 month, like [1.23, 4.56]
NEW_PPM = []     # peaks at 1 month that were not there at 1 day
KEPT_PPM = []    # peaks there at both times

who = nt.io.NAMES[sample]
nt.record.write_down("LOST_PPM", LOST_PPM, who, "lost_ppm", suggestion=changes["lost"])
nt.record.write_down("NEW_PPM", NEW_PPM, who, "new_ppm", suggestion=changes["new"])
nt.record.write_down("KEPT_PPM", KEPT_PPM, who, "kept_ppm", suggestion=changes["kept"]);

## Part 4: Verdict and comparison

### What you need

- **Survives:** every signal from hydrogens on carbon stays put.
- **Side chain changed:** the backbone signal stays, and new signals grow beside it.
- **Breaks down:** the backbone signal goes.

If the side-chain amide turned into an acid group, asparagine would become aspartic acid and
glutamine would become glutamic acid. Look closer, below, lays that amino acid over your spectrum.

Longer version: NMR project step 2, Can a molecule change but not fall apart? Find the change in an amino acid's spectra.

> **RECORD YOUR ANSWER: your call on your amino acid**
>
> **What gets saved:** `CALL`, `PRODUCT`, `CONFIDENCE` and `NOTE`, four short answers in quotes.
>
> **Where to look:** your backbone signal and the lost, new and kept peaks you saved above.
>
> **How to fill this in:**
> - `CALL`: `"survives"`, `"side chain changed"`, `"breaks down"` or `"unsure"` (unsure is a good answer when the data do not settle it)
> - `PRODUCT`: what the side chain may have become, in words; `"none"`; or `""` if you have no idea yet
> - `CONFIDENCE`: `"low"`, `"medium"` or `"high"`
> - `NOTE`: anything you want your reviewer to know, or `""`

<details><summary>Stuck? Open hint 1</summary>

Is your backbone signal in KEPT_PPM? Is anything in NEW_PPM?

</details>

<details><summary>Stuck? Open hint 2</summary>

Backbone kept and new signals growing: side chain changed. Backbone kept and nothing new:
survives. Backbone gone: breaks down.

</details>

In [ ]:
CALL = ""          # "survives", "side chain changed", "breaks down" or "unsure"
PRODUCT = ""       # what the side chain may have become, "none", or "" if no idea yet
CONFIDENCE = ""    # "low", "medium" or "high"
NOTE = ""          # anything you want your reviewer to know

nt.record.write_down("CALL", CALL, who, "call")
nt.record.write_down("PRODUCT", PRODUCT, who, "product_guess")
nt.record.write_down("CONFIDENCE", CONFIDENCE, who, "confidence")
nt.record.write_down("NOTE", NOTE, who, "note");

### Your verdict

Here is your verdict, built from what you recorded above. Change any line above and re-run this
cell to update it. The alanine row is the filled example from Part 1.

In [ ]:
MY_VERDICT = nt.record.build_verdict(
    {CASE: {"call": CALL, "lost_ppm": LOST_PPM, "new_ppm": NEW_PPM, "kept_ppm": KEPT_PPM,
            "product_guess": PRODUCT, "confidence": CONFIDENCE, "note": NOTE}},
    names={CASE: f"{who}, {nt.io.ACID_PCT[acid]}"}, examples={"alanine, 98% (example)": ALA_EXAMPLE})[CASE]

The next cell compares your verdict with the paper where it can. Every amino acid case here is
checked against Table 1 of the paper (M. D. Seager et al. 2024, *Astrobiology* 24, 386), so read
it there too.

In [ ]:
feedback = nt.checks.Report()
if nt.record.revealed("amino-acid"):
    key = nt.answers.load("amino_acids")
    c = nt.answers.case(key, CASE)
    feedback = nt.checks.check_verdict(MY_VERDICT, c).show("Your verdict")

### Look closer (optional): test one idea

If the side-chain amide turned into an acid, asparagine would become aspartic acid and glutamine
would become glutamic acid. The next cell lays your 1 month spectrum over that amino acid, recorded
in the same acid. Do any new peaks of yours line up with it? Then check your call against Table 1
of the paper (M. D. Seager et al. 2024, *Astrobiology* 24, 386).

In [ ]:
product = {"ASN": "ASP", "GLN": "GLU"}[sample]
prod = nt.load_series("8381013", product, acid=acid, nucleus="1H", times=["1month"])[0]
fig_prod, ax = nt.plot.stack([prod, last], 3.0, 6.5, norm=(3.0, 6.5),
                             labels=[f"{nt.io.NAMES[product]}, 1 month", f"{nt.io.NAMES[sample]}, 1 month"])
plt.show()

## Hand-in

> **SAVE YOUR HAND-IN: your amino acid hand-in**
>
> **What gets saved:** in `handin/`: your stack and `verdict.json` (your backbone signal, verdict, peak lists and feedback).
>
> **Where to look:** nothing new to fill in: the cell uses what you recorded above.
>
> **How to fill this in:**
> - Run the cell. Nothing is sent for review until you run `nt.portal.submit()`.

<details><summary>Stuck? Open hint 1</summary>

If the figure is missing from the list it prints, run the cell that draws it, then this cell again.

</details>

<details><summary>Stuck? Open hint 2</summary>

If it says a name is not defined, run the notebook from the top (Runtime > Run all in Colab).

</details>

In [ ]:
record = nt.record.handin(
    "amino-acid-verdict", figures={"stack_" + CASE: fig_case},
    verdict={"case": CASE, "backbone_ppm": None if BACKBONE_PPM is ... else BACKBONE_PPM, "verdict": MY_VERDICT,
             "my_peaks_ppm": {t: nt.peaks.ppms(v) for t, v in my_peaks.items()},
             "feedback": [str(f) for f in feedback]},
    numbers={"case": CASE, "call": MY_VERDICT["call"] or "blank", "product": MY_VERDICT["product_guess"] or "blank"},
    question="amino-acid", traps=nt.checks.traps_flagged(traps))
# nt.portal.submit()